In [13]:
#download needed libraries,
%pip install kagglehub numpy pandas scikit-learn


     ---------------------------------------- 0.0/61.0 kB ? eta -:--:--
     ------ --------------------------------- 10.2/61.0 kB ? eta -:--:--
     ------------ ------------------------- 20.5/61.0 kB 217.9 kB/s eta 0:00:01
     -------------------------------------- 61.0/61.0 kB 465.2 kB/s eta 0:00:00
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
    --------------------------------------- 0.2/8.3 MB 4.1 MB/s eta 0:00:02
   -- ------------------------------------- 0.4/8.3 MB 4.6 MB/s eta 0:00:02
   --- ------------------------------------ 0.7/8.3 MB 5.9 MB/s eta 0:00:02
   ----- ---------------------------------- 1.1/8.3 MB 5.8 MB/s eta 0:00:02
   ----- ---------------------------------- 1.2/8.3 MB 5.9 MB/s eta 0:00:02
   ------- -------------------------------- 1.5/8.3 MB 5.4 MB/s eta 0:00:02
   -------- ------------------------------- 1.7/8.3 MB 5.3 MB/s eta 0:00:02
   --------- ------------


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: C:\Users\AVENg\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:

#load dataset
df = pd.read_csv('.\DATA\Airbnb_Open_Data.csv')

C:\Users\AVENg\AppData\Local\Temp\ipykernel_19584\1483286757.py:2: DtypeWarning: Columns (0: license) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('.\DATA\Airbnb_Open_Data.csv')


In [5]:
import pandas as pd
import numpy as np

# try to get the file or crash out if missing
try:
    df = pd.read_csv('.\DATA\Airbnb_Open_Data.csv')
    print("row count:", len(df))
except:
    print("fix path to Airbnb_Open_Data.csv")

# lower case cols and add underscores
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

# clean up the messy price columns 
for c in ['price', 'service_fee']:
    if c in df.columns:
        df[c] = df[c].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False).str.strip()
        df[c] = pd.to_numeric(df[c], errors='coerce')

print("\ncols ready:")
print(df[['price', 'service_fee']].dtypes)

C:\Users\AVENg\AppData\Local\Temp\ipykernel_19584\1233052087.py:6: DtypeWarning: Columns (0: license) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('.\DATA\Airbnb_Open_Data.csv')


row count: 102599

cols ready:
price          float64
service_fee    float64
dtype: object


In [6]:
# stick medians into numerical blanks
med_price = df['price'].median()
df['price'] = df['price'].fillna(med_price)
df['service_fee'] = df['service_fee'].fillna(df['service_fee'].median())
df['construction_year'] = df['construction_year'].fillna(df['construction_year'].median())

# fix neighborhood typos and blanks
if 'neighbourhood_group' in df.columns:
    df['neighbourhood_group'] = df['neighbourhood_group'].fillna('Unknown').replace({'brookln': 'Brooklyn', 'manhatan': 'Manhattan'})

# fill policy and room types with mode
df['cancellation_policy'] = df['cancellation_policy'].fillna(df['cancellation_policy'].mode()[0])
df['room_type'] = df['room_type'].fillna(df['room_type'].mode()[0])

print("nulls left:")
print(df[['price', 'service_fee', 'construction_year', 'neighbourhood_group']].isnull().sum())

nulls left:
price                  0
service_fee            0
construction_year      0
neighbourhood_group    0
dtype: int64


In [ ]:
#remove rows with negative or zer0 values
bad_rows = df[df['price'] <= 0].index
df = df.drop(index=bad_rows).reset_index(drop=True)
print(f"dropped {len(bad_rows)} zero-price rows")